In [ ]:
#organiser et fusionner des fichiers CSV de données financières
import pandas as pd
import os
from glob import glob
import numpy as np

# --- 1. Définir les dossiers où sont les fichiers CSV ---
prices_folder = "data/prices"          
fundamentals_folder = "data/fundamentals"  

# --- 2. Lire et concaténer tous les fichiers prices ---
price_files = glob(os.path.join(prices_folder, "*.csv"))

prices_list = []
for file in price_files:
    # Lire le CSV en ignorant les lignes inutiles
    df = pd.read_csv(file, skiprows=[1, 2])  # ignore Ticker et Date,,,,
    
    # Renommer la première colonne si nécessaire
    if df.columns[0] != "Date":
        df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    
    # Convertir la colonne Date en datetime
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Ajouter le ticker depuis le nom du fichier
    ticker = os.path.basename(file).split(".")[0]
    df["Ticker"] = ticker
    
    prices_list.append(df)

prices_df = pd.concat(prices_list, ignore_index=True)
print("Prices dataset shape:", prices_df.shape)

# --- 3. Lire et concaténer tous les fichiers fundamentals ---
fund_files = glob(os.path.join(fundamentals_folder, "*.csv"))

funds_list = []
for file in fund_files:
    # Lire le CSV en ignorant les lignes inutiles
    df = pd.read_csv(file, skiprows=[1, 2], low_memory=False)  # skip Ticker et ligne vide
    
    # Renommer la première colonne si nécessaire
    if df.columns[0] != "Date":
        df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    
    # Convertir la colonne Date en datetime
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Ajouter le ticker depuis le nom du fichier
    ticker = os.path.basename(file).split("_")[0]  # ex: 'AAPL_fundamentals.csv'
    df["Ticker"] = ticker
    
    funds_list.append(df)

fundamentals_df = pd.concat(funds_list, ignore_index=True)
print("Fundamentals dataset shape:", fundamentals_df.shape)

# Pour prices_df
cols = ["Ticker"] + [c for c in prices_df.columns if c != "Ticker"]
prices_df = prices_df[cols]

# Pour fundamentals_df
cols = ["Ticker"] + [c for c in fundamentals_df.columns if c != "Ticker"]
fundamentals_df = fundamentals_df[cols]

print(prices_df.head())

print(fundamentals_df.head())

# --- 4. Sauvegarder les datasets fusionnés ---
prices_df.to_csv("all_prices.csv", index=False)
fundamentals_df.to_csv("all_fundamentals.csv", index=False)


In [ ]:
#nettoyer les données:
# Supprimer la colonne Close puisque Close_Adj est plus pertinente ( elle tient compte des dividendes et fractionnements d'actions)
if "Close" in prices_df.columns:
    prices_df.drop(columns=["Close"], inplace=True)

# Réordonner les colonnes pour mettre Ticker en premier
cols = ["Ticker"] + [c for c in prices_df.columns if c != "Ticker"]
prices_df = prices_df[cols]
print(prices_df.head())


# --- Filtrer les données à partir de 2024 pour travailler sur des données recentes ---
prices_df = prices_df[prices_df["Date"] >= "2024-01-01"]
print("Filtered Prices dataset shape:", prices_df.shape)


#nettoyer fundamentals_df en gardant uniquement les colonnes essentielles pour calculer des ratios financiers
# Colonnes essentielles pour les ratios fondamentaux
cols_to_keep = [
    # Identifiants
    "Ticker", "Date",
    
    # Revenus et bénéfices
    "Total Revenue", "Operating Revenue", "EBITDA", "EBIT", "Operating Income",
    "Net Income", "Net Income From Continuing Operations", "Net Income Common Stockholders",
    
    # Actions et EPS
    "Diluted Average Shares", "Basic Average Shares", "Diluted EPS", "Basic EPS",
    
    # Bilan
    "Total Debt", "Net Debt", "Current Assets", "Current Liabilities", "Cash Cash Equivalents And Short Term Investments",
    "Accounts Receivable", "Inventory", "Invested Capital", "Total Equity", "Common Stock Equity",
    
    # Coût
    "Cost Of Revenue"
]
# Sélectionner uniquement les colonnes qui existent dans le DataFrame fundamentals_df
cols_to_keep_existing = [col for col in cols_to_keep if col in fundamentals_df.columns]

# Conserver uniquement les colonnes nécessaires
fundamentals_df_clean = fundamentals_df[cols_to_keep_existing]

fundamentals_df=fundamentals_df_clean

print("Cleaned Fundamentals dataset shape:", fundamentals_df.shape)

# --- 4. Sauvegarder  ---
prices_df.to_csv("all_prices.csv", index=False)
fundamentals_df.to_csv("all_fundamentals.csv", index=False)


In [ ]:
# lire les données dans prices_2025 pour recuperer les données de 2025 ( prices_df contient actuellement les données de 2024 seulement )

prices_folder_2025 = "data/prices_2025"          

# --- 2. Lire et concaténer tous les fichiers prices ---
price_files_2025 = glob(os.path.join(prices_folder_2025, "*.csv"))

prices_list_2025 = []#une liste de dataframes
for file in price_files_2025:
    # Lire le CSV en ignorant les lignes inutiles
    df = pd.read_csv(file, skiprows=[1, 2])  # ignore Ticker et Date,,,,
    
    # Renommer la première colonne si nécessaire
    if df.columns[0] != "Date":
        df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    
    # Convertir la colonne Date en datetime
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Ajouter le ticker depuis le nom du fichier
    ticker = os.path.basename(file).split(".")[0]
    df["Ticker"] = ticker
    
    prices_list_2025.append(df)

prices_df_2025 = pd.concat(prices_list_2025, ignore_index=True)
print("Prices dataset shape:", prices_df_2025.shape)

# 1. Supprimer la colonne 'Close' de prices_df_2025 si elle existe
if 'Close' in prices_df_2025.columns:
    prices_df_2025 = prices_df_2025.drop(columns=['Close'])

# 2. Filtrer pour ne garder que les dates >= 2024
prices_df_2025['Date'] = pd.to_datetime(prices_df_2025['Date'])
prices_df_2025 = prices_df_2025[prices_df_2025['Date'].dt.year >= 2024]

# 3. Concaténer avec prices_df
combined_prices = pd.concat([prices_df, prices_df_2025], ignore_index=True)

# 4. Supprimer les doublons basés sur Ticker + Date
combined_prices = combined_prices.drop_duplicates(subset=['Ticker', 'Date'], keep='first')

# 5. Trier par Ticker puis Date
combined_prices = combined_prices.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

# 6. Mettre à jour prices_df
prices_df = combined_prices

print("Updated prices_df shape:", prices_df.shape)

#enregistrer le nouveau fichier CSV avec les données mises à jour
prices_df.to_csv("all_prices.csv", index=False)

#en effet maintenant chaque jour dans prices ( 2024 ou 2025) peut trouver le rapport fondamental le plus récent dans fundamentals_df (exemple: pour un rapport ds fundamentals mis dans decembre, il sera utilisé pour tous les jours de dec, jan et fev jusqu'au prochain rapport fondamental en mars)

In [ ]:
#ajouter les indicateurs exploitables par le modele dans prices_df


# Normaliser le nom de la colonne ajustée
if 'Adj Close' in prices_df.columns:
    prices_df.rename(columns={'Adj Close': 'Close_Adj'}, inplace=True)
else:
    raise ValueError("No adjusted close column found")

print(prices_df[['Open', 'Close_Adj']].dtypes)
prices_df['Open'] = pd.to_numeric(prices_df['Open'], errors='coerce') # Convertir en numérique avec gestion des erreurs

# Daily return
prices_df['Daily_Return'] = (prices_df['Close_Adj'] - prices_df['Open']) / prices_df['Open'] * 100

# Volatility
prices_df['Volatility'] = prices_df.groupby('Ticker')['Daily_Return'].rolling(window=20).std().reset_index(0, drop=True)
prices_df['Annual_Volatility'] = prices_df['Volatility'] * np.sqrt(252)

# Moving Averages
for ma in [10, 50, 200]:
    prices_df[f'MA{ma}'] = prices_df.groupby('Ticker')['Close_Adj'].transform(lambda x: x.rolling(ma).mean())

# RSI
N = 14
def compute_rsi(x):
    delta = x.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(N).mean()
    avg_loss = loss.rolling(N).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

prices_df['RSI'] = prices_df.groupby('Ticker')['Close_Adj'].transform(compute_rsi)

# MACD
def compute_macd(x):
    ema12 = x.ewm(span=12, adjust=False).mean()
    ema26 = x.ewm(span=26, adjust=False).mean()
    return ema12 - ema26

prices_df['MACD'] = prices_df.groupby('Ticker')['Close_Adj'].transform(compute_macd)

# Drawdown
def compute_drawdown(x):
    roll_max = x.cummax()
    drawdown = (x - roll_max) / roll_max
    return drawdown

prices_df['Drawdown'] = prices_df.groupby('Ticker')['Close_Adj'].transform(compute_drawdown)

# Momentum features
for n in [1, 5, 10, 20]:
    prices_df[f'Momentum_{n}D'] = prices_df.groupby('Ticker')['Close_Adj'].transform(lambda x: x - x.shift(n))

#lag features
for n in [1, 5, 10]:
    prices_df[f'Return_Lag_{n}D'] = prices_df.groupby('Ticker')['Daily_Return'].shift(n)
#features temporaires / saisonnieres:
prices_df['Day_of_Week'] = prices_df['Date'].dt.dayofweek
prices_df['Month'] = prices_df['Date'].dt.month
prices_df['Quarter'] = prices_df['Date'].dt.quarter

#les features fond+prices:
# --- 1. Assurer que les dates sont du type datetime et sans NaN ---
prices_df['Date'] = pd.to_datetime(prices_df['Date'], errors='coerce')
fundamentals_df['Date'] = pd.to_datetime(fundamentals_df['Date'], errors='coerce')

# Supprimer les lignes où Date est NaT
prices_df = prices_df.dropna(subset=['Date'])
fundamentals_df = fundamentals_df.dropna(subset=['Date'])

# --- 2. Trier les DataFrames par Ticker puis Date ---
prices_df = prices_df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)
fundamentals_df = fundamentals_df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)



# Sauvegarder les données mises à jour
prices_df.to_csv("all_prices.csv", index=False)


In [ ]:

#fusionner les deux datasets prices et fundamentals
# Associer chaque ligne de prices avec le dernier rapport fondamental disponible

# Trier par Ticker puis Date si ce n'est pas déjà fait
prices_df = prices_df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)
fundamentals_df = fundamentals_df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

# On merge "asof" par ticker (le rapport fondamental le plus récent <= date du prix)
merged_list = []
for ticker in prices_df['Ticker'].unique():
    prices_ticker = prices_df[prices_df['Ticker'] == ticker].copy()
    funds_ticker = fundamentals_df[fundamentals_df['Ticker'] == ticker].copy()
    
    if not funds_ticker.empty:
        merged = pd.merge_asof(
            prices_ticker,
            funds_ticker,
            on='Date',
            direction='backward',  # prendre le dernier rapport fondamental <= date
            tolerance=pd.Timedelta('92D')  # max 3 mois
        )
        merged_list.append(merged)
    else:
        merged_list.append(prices_ticker)

merged_df = pd.concat(merged_list, ignore_index=True)
print("Merged dataset shape:", merged_df.shape)

# Sauvegarder le dataset fusionné
merged_df.to_csv("merged_prices_fundamentals.csv", index=False)


In [14]:
#nettoyer merged_df en supprimant les lignes avec des valeurs manquantes dans des colonnes critiques pour calculer des ratios financiers
# Colonnes critiques pour les ratios fondamentaux et basés sur les prix
critical_columns = [
    # Fondamentaux
    'Net Income Common Stockholders', 'Common Stock Equity', 'Net Income', 'Total Revenue',
    'Total Debt', 'Current Assets', 'Current Liabilities', 'Inventory', 'Operating Income',
    'Diluted EPS',
    # Prix
    'Close_Adj'
]

# Supprimer les lignes où l'une de ces colonnes est NaN
merged_df = merged_df.dropna(subset=critical_columns)

# --- Nettoyer les colonnes Ticker après merge ---
if 'Ticker_x' in merged_df.columns:
    merged_df.rename(columns={'Ticker_x': 'Ticker'}, inplace=True)

# Supprimer les doublons de colonnes Ticker
if 'Ticker_y' in merged_df.columns:
    merged_df.drop(columns=['Ticker_y'], inplace=True)

# Supprimer la deuxième colonne 'Ticker' si elle existe à la fin
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

# Vérifier qu'il ne reste qu'une seule colonne Ticker
print(merged_df.columns)


#enregistrer le dataset nettoyé
merged_df.to_csv("merged_prices_fundamentals.csv", index=False)

Index(['Ticker', 'Date', 'Close_Adj', 'High', 'Low', 'Open', 'Volume',
       'Daily_Return', 'Volatility', 'Annual_Volatility', 'MA10', 'MA50',
       'MA200', 'RSI', 'MACD', 'Drawdown', 'Momentum_1D', 'Momentum_5D',
       'Momentum_10D', 'Momentum_20D', 'Return_Lag_1D', 'Return_Lag_5D',
       'Return_Lag_10D', 'Day_of_Week', 'Month', 'Quarter', 'Total Revenue',
       'Operating Revenue', 'EBITDA', 'EBIT', 'Operating Income', 'Net Income',
       'Net Income From Continuing Operations',
       'Net Income Common Stockholders', 'Diluted Average Shares',
       'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Total Debt',
       'Net Debt', 'Current Assets', 'Current Liabilities',
       'Cash Cash Equivalents And Short Term Investments',
       'Accounts Receivable', 'Inventory', 'Invested Capital',
       'Common Stock Equity', 'Cost Of Revenue', 'ROE', 'Profit_Margin',
       'Debt_to_Equity', 'EPS_Growth', 'Revenue_Growth', 'Current_Ratio',
       'Quick_Ratio', 'Operating_Ma

In [ ]:


#calculer les ratios financiers dans le dataset fusionné merged_df

merged_df['ROE'] = merged_df['Net Income Common Stockholders'] / merged_df['Common Stock Equity']
merged_df['Profit_Margin'] = merged_df['Net Income'] / merged_df['Total Revenue']
merged_df['Debt_to_Equity'] = merged_df['Total Debt'] / merged_df['Common Stock Equity']

merged_df['EPS_Growth'] = merged_df.groupby('Ticker')['Diluted EPS'].pct_change()
merged_df['Revenue_Growth'] = merged_df.groupby('Ticker')['Total Revenue'].pct_change()
merged_df['Current_Ratio'] = merged_df['Current Assets'] / merged_df['Current Liabilities']
merged_df['Quick_Ratio'] = (merged_df['Current Assets'] - merged_df['Inventory']) / merged_df['Current Liabilities']
merged_df['Operating_Margin'] = merged_df['Operating Income'] / merged_df['Total Revenue']

# Pour les ratios basés sur les prix
merged_df['P_to_E'] = merged_df['Close_Adj'] / merged_df['Diluted EPS']
merged_df['Price_to_Book'] = merged_df['Close_Adj'] / merged_df['Common Stock Equity']
merged_df['Price_to_Revenue'] = merged_df['Close_Adj'] / merged_df['Total Revenue']



print(merged_df[['Ticker','Date','ROE','Profit_Margin','Debt_to_Equity','P_to_E']].head())


# Sauvegarder le dataset final avec les ratios financiers
merged_df.to_csv("merged_prices_fundamentals.csv", index=False)

Index(['Ticker', 'Date', 'Close_Adj', 'High', 'Low', 'Open', 'Volume',
       'Daily_Return', 'Volatility', 'Annual_Volatility', 'MA10', 'MA50',
       'MA200', 'RSI', 'MACD', 'Drawdown', 'Momentum_1D', 'Momentum_5D',
       'Momentum_10D', 'Momentum_20D', 'Return_Lag_1D', 'Return_Lag_5D',
       'Return_Lag_10D', 'Day_of_Week', 'Month', 'Quarter', 'Total Revenue',
       'Operating Revenue', 'EBITDA', 'EBIT', 'Operating Income', 'Net Income',
       'Net Income From Continuing Operations',
       'Net Income Common Stockholders', 'Diluted Average Shares',
       'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Total Debt',
       'Net Debt', 'Current Assets', 'Current Liabilities',
       'Cash Cash Equivalents And Short Term Investments',
       'Accounts Receivable', 'Inventory', 'Invested Capital',
       'Common Stock Equity', 'Cost Of Revenue', 'ROE', 'Profit_Margin',
       'Debt_to_Equity', 'EPS_Growth', 'Revenue_Growth', 'Current_Ratio',
       'Quick_Ratio', 'Operating_Ma